# Buổi 13 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `feature.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Bộ feature và bảng biết trước (mục 4.1, 4.2, 4.4)

In [ ]:
%matplotlib inline
import warnings

import feature as ft
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
sau = pd.Series([10, 12, 8, 14, 20, 16.0], index=range(1, 7))
nhom = pd.Series(list("ABABAB"), index=sau.index)
thieu = sau.copy()
thieu[3] = np.nan
print(pd.DataFrame({
    "y": sau,
    "rolling(3, center)": sau.rolling(3, center=True).mean(),
    "rolling(3)": sau.rolling(3).mean(),
    "shift(1).rolling(3)": sau.shift(1).rolling(3).mean(),
    "z toàn chuỗi": (sau - sau.mean()) / sau.std(),
    "TB nhóm toàn chuỗi": sau.groupby(nhom).transform("mean"),
    "TB nhóm ngày đã qua": sau.groupby(nhom).transform(lambda v: v.shift(1).expanding().mean()),
    "nội suy hai phía": thieu.interpolate(limit_direction="both"),
}).round(2))

y = ft.doc_ban_le().ffill()
f = ft.bo_feature(y)
print(f.shape)
bang = ft.bang_biet_truoc(f)
print(bang["biết trước"].value_counts().to_dict())
print(bang[bang["feature"].isin(["tb_7", "z_score", "tb_theo_thu", "y_dien_hai_chieu"])].to_string(index=False))

## Bước 2 — Tự viết `kiem_ro_ri` (mục 4.5)

Sửa `kiem_ro_ri` rồi chạy lại ô này.

In [ ]:
print(ft.kiem_ro_ri(ft.bo_feature, y))

## Bước 3 — Sửa bốn feature rò rỉ (mục 4.2, 4.4)

Sửa `feature_tre` rồi chạy lại ô này.

In [ ]:
f = ft.bo_feature(y)
print("số feature:", f.shape[1])
print("cắt tương lai:", ft.kiem_ro_ri(ft.bo_feature, y)["feature"].tolist())
print("nhiễu mục tiêu:", ft.kiem_nhieu_muc_tieu(ft.bo_feature, y))

## Bước 4 — Tết cho mọi năm (mục 4.3)

Sửa `feature_lich` rồi chạy lại ô này.

In [ ]:
lech = []
for nam in range(2000, 2036):
    le = holidays.country_holidays("VN", years=nam)
    if ft.tet(nam) not in {d for d, ten in le.items() if ten == "Lunar New Year"}:
        lech.append(nam)
print("năm lệch holidays:", lech, "| Tết 2007 (UTC+7, UTC+8):", ft.tet(2007, 7), ft.tet(2007, 8))
d = ft.feature_lich(pd.date_range("2016-01-01", "2025-12-31"))["so_ngay_toi_tet"]
print("ngày có so_ngay_toi_tet = 0:", [str(x.date()) for x in d[d == 0].index])
print(ft.gia_tri_feature_tet().round(1).to_string(index=False))

## Bước 5 — Giá của rò rỉ (mục 4.6)

Không cần sửa. Cuối cùng: `python lab.py check` trong terminal.

In [ ]:
bang = ft.gia_cua_ro_ri()
print(bang.round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(bang["bộ feature"], bang["so với chỉ lag (%)"])
ax.axvline(0, color="black")
ax.set_xlabel("MAE so với chỉ lag (%)");